In [6]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from tqdm import tqdm
import re

In [7]:
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)


True

In [8]:
stop_words = set(stopwords.words("english"))
lemma = WordNetLemmatizer()
tqdm.pandas()

In [9]:
def load_and_preprocess(file_path, limit=2000):
    """
    Load JSON dataset, filter for at least 3 taxonomy levels,
    split into Level1–Level4, clean text, and preserve Level4 for later.
    """
    # 1️⃣ Load JSON file (limit rows for faster testing)
    df = pd.read_json(file_path).head(limit)

    # 2️⃣ Keep only rows with >= 2 '>' → means Level1, Level2, Level3 exist
    df = df[df['pathlist_names'].str.count('>') >= 2].reset_index(drop=True)
    df.fillna("", inplace=True)  # replace NaNs with empty strings

    # 3️⃣ Keep only relevant columns for processing
    cols = [
        "Title", "BrandInfo.BrandName", "ProductName", "Category.Name.Value",
        "SummaryDescription.LongSummaryDescription",
        "SummaryDescription.ShortSummaryDescription",
        "Description.LongProductName", "Description.LongDesc",
        "pathlist_names"
    ]
    df = df.reindex(columns=cols, fill_value="")  # fill missing with empty string

    # 4️⃣ Split taxonomy into Level1–Level4
    path_split = df['pathlist_names'].str.split('>', expand=True)
    df['Level1'] = path_split[0].str.strip()
    df['Level2'] = path_split[1].str.strip()
    df['Level3'] = path_split[2].str.strip()
    df['Level4'] = path_split[3].str.strip() if path_split.shape[1] > 3 else ""

    # 5️⃣ Combine descriptive text fields into one column
    text_cols = [
        "Title", "BrandInfo.BrandName", "ProductName", "Category.Name.Value",
        "SummaryDescription.LongSummaryDescription",
        "SummaryDescription.ShortSummaryDescription",
        "Description.LongProductName", "Description.LongDesc"
    ]
    df["raw_text"] = df[text_cols].agg(" ".join, axis=1)

    # 6️⃣ Text cleaning function
    def clean_text(text):
        # keep alphanumeric + '-' and '+' (so product codes remain)
        tokens = [t.lower() for t in word_tokenize(text) if re.match(r"^[A-Za-z0-9\-\+]+$", t)]
        tokens = [lemma.lemmatize(t) for t in tokens if t not in stop_words]
        return " ".join(tokens)

    # 7️⃣ Apply cleaning to raw_text
    df["cleaned_text"] = df["raw_text"].progress_apply(clean_text)

    return df


In [10]:
# Process train/validation/test datasets
# ---------------------------
df_train = load_and_preprocess("icecat_data_train.json")
df_validate = load_and_preprocess("icecat_data_validate.json")
df_test = load_and_preprocess("icecat_data_test.json")

100%|██████████| 2000/2000 [00:05<00:00, 358.15it/s]


In [11]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 15 columns):
 #   Column                                      Non-Null Count  Dtype 
---  ------                                      --------------  ----- 
 0   Title                                       2000 non-null   object
 1   BrandInfo.BrandName                         2000 non-null   object
 2   ProductName                                 2000 non-null   object
 3   Category.Name.Value                         2000 non-null   object
 4   SummaryDescription.LongSummaryDescription   2000 non-null   object
 5   SummaryDescription.ShortSummaryDescription  2000 non-null   object
 6   Description.LongProductName                 2000 non-null   object
 7   Description.LongDesc                        2000 non-null   object
 8   pathlist_names                              2000 non-null   object
 9   Level1                                      2000 non-null   object
 10  Level2                  

In [12]:
df_train

,Title,BrandInfo.BrandName,ProductName,Category.Name.Value,SummaryDescription.LongSummaryDescription,SummaryDescription.ShortSummaryDescription,Description.LongProductName,Description.LongDesc,pathlist_names,Level1,Level2,Level3,Level4,raw_text,cleaned_text
0,ASUS K31CD-IT049T PC 6th gen Intel® Core™ i7 i...,ASUS,K31CD-IT049T,PCs/Workstations,ASUS K31CD-IT049T. Processor frequency: 3.4 GH...,"ASUS K31CD-IT049T, 3.4 GHz, 6th gen Intel® Cor...","Intel Core i7-6700 (8M Cache, 3.4GHz), 16GB RA...",<b>Smart Multimedia Performance</b><br>\nVivoP...,Computers & Electronics>Computers>PCs/Workstat...,Computers & Electronics,Computers,PCs/Workstations,None,ASUS K31CD-IT049T PC 6th gen Intel® Core™ i7 i...,asus k31cd-it049t pc 6th gen i7 i7-6700 16 gb ...
1,HP 686915-A41 notebook spare part Keyboard,HP,686915-A41,Notebook Spare Parts,HP 686915-A41. Type: Keyboard. Keyboard langua...,"HP 686915-A41, Keyboard, Belgian, Keyboard bac...",Keyboard in midnight black finish with backlig...,,Computers & Electronics>Computers>Notebook Par...,Computers & Electronics,Computers,Notebook Parts & Accessories,Notebook Spare Parts,HP 686915-A41 notebook spare part Keyboard HP ...,hp 686915-a41 notebook spare part keyboard hp ...
2,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,C2G,1m ST/SC Plenum-Rated 9/125 Duplex Single-Mode...,Fibre Optic Cables,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,1m ST/SC Plenum-Rated 9/125 Duplex Single-Mode...,Get the performance you demand at a price that...,Computers & Electronics>Computer Cables>Fibre ...,Computers & Electronics,Computer Cables,Fibre Optic Cables,None,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,c2g 1m plenum-rated duplex single-mode fiber p...
3,HP FA889AA Battery,HP,FA889AA,Handheld Mobile Computer Spare Parts,"HP FA889AA. Product type: Battery, Product col...","HP FA889AA, Battery, White, Lithium-Ion (Li-Io...","1100 mAh, Lithium Ion, Standard Battery",Keeping an extra source of power nearby means ...,Computers & Electronics>Computers>Handheld Mob...,Computers & Electronics,Computers,Handheld Mobile Computer Spare Parts,None,HP FA889AA Battery HP FA889AA Handheld Mobile ...,hp fa889aa battery hp fa889aa handheld mobile ...
4,Lenovo ThinkStation C30 Intel® Xeon® E5 Family...,Lenovo,C30,PCs/Workstations,Lenovo ThinkStation C30. Processor frequency: ...,"Lenovo ThinkStation C30, 2 GHz, Intel® Xeon® E...","Intel Xeon E5-2620 (15M Cache, 2.00 GHz, 7.20 ...",The C30 builds on its award-winning design as ...,Computers & Electronics>Computers>PCs/Workstat...,Computers & Electronics,Computers,PCs/Workstations,None,Lenovo ThinkStation C30 Intel® Xeon® E5 Family...,lenovo thinkstation c30 e5 family e5-2620 4 gb...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,HP L32629-601 notebook spare part Motherboard,HP,L32629-601,Notebook Spare Parts,"HP L32629-601. Type: Motherboard, Brand compat...","HP L32629-601, Motherboard, HP, 17",System board,For use in models with discrete graphics memor...,Computers & Electronics>Computers>Notebook Par...,Computers & Electronics,Computers,Notebook Parts & Accessories,Notebook Spare Parts,HP L32629-601 notebook spare part Motherboard ...,hp l32629-601 notebook spare part motherboard ...
1996,B-Tech BT8430 flat panel wall mount 119.4 cm (...,B-Tech,BT8430,Flat Panel Wall Mounts,"B-Tech BT8430. Suitable for: TV, Maximum weigh...","B-Tech BT8430, TV, 50 kg, 119.4 cm (47""), 75 x...","119.38 cm (47"") max, 50kg, VESA 200 x 200 max,...","Designed to mount medium sized screens, the BT...",Computers & Electronics>TVs & Monitors>Flat Pa...,Computers & Electronics,TVs & Monitors,Flat Panel Wall Mounts,None,B-Tech BT8430 flat panel wall mount 119.4 cm (...,b-tech bt8430 flat panel wall mount cm 47 blac...
1997,"Trend Micro NeatSuite Advanced, 11m, 510-750u,...",Trend Micro,"NeatSuite Advanced, 11m, 510-750u, EN",Software Licenses/Upgrades,"Trend Micro NeatSuite Advanced, 11m, 510-750u,...","Trend Micro Ne

In [13]:
# 8️⃣ Save processed outputs with Level4 preserved
df_train.to_json("phase1_train_with_L4.json", orient="records", lines=True)
df_validate.to_json("phase1_validate_with_L4.json", orient="records", lines=True)
df_test.to_json("phase1_test_with_L4.json", orient="records", lines=True)

In [14]:
print("✅ Phase 1 complete: Cleaned datasets with Level4 preserved")
print("Sample:\n", df_train[['pathlist_names', 'Level1', 'Level2', 'Level3', 'Level4']].head())

✅ Phase 1 complete: Cleaned datasets with Level4 preserved
Sample:
                                       pathlist_names                   Level1  \
0  Computers & Electronics>Computers>PCs/Workstat...  Computers & Electronics   
1  Computers & Electronics>Computers>Notebook Par...  Computers & Electronics   
2  Computers & Electronics>Computer Cables>Fibre ...  Computers & Electronics   
3  Computers & Electronics>Computers>Handheld Mob...  Computers & Electronics   
4  Computers & Electronics>Computers>PCs/Workstat...  Computers & Electronics   

            Level2                                Level3                Level4  
0        Computers                      PCs/Workstations                  None  
1        Computers          Notebook Parts & Accessories  Notebook Spare Parts  
2  Computer Cables                    Fibre Optic Cables                  None  
3        Computers  Handheld Mobile Computer Spare Parts                  None  
4        Computers                      